# README figures

Generates the five figures embedded in the repository README. Nothing here is new
analysis — each panel re-draws a result already computed in [`EDA.ipynb`](EDA.ipynb),
and prints the numbers the README captions quote so the two can be checked against
each other.

Run from inside `notebooks/` (the data path is relative to the working directory):

```sh
cd notebooks && uv run jupyter lab
```

then *Run All*. The PNG files land in `../figures/` and are committed, so a reader
without the workbook still sees them. Every number printed below is the same value as
in the corresponding `EDA.ipynb` section; if one drifts, the figure is stale.

| figure | source section |
| --- | --- |
| `01_units_vs_spend.png` | EDA §2, weekly totals and trend tests |
| `02_promotion_economics.png` | EDA §5.2, discount against incremental spend |
| `03_basket_economics.png` | EDA §5.3, spend per basket and units per basket |
| `04_household_cost.png` | EDA §5.5, cost per incremental household |
| `05_seasonality.png` | EDA §3, seasonal profile and permutation test |

Lift is reported as a **ratio** (1.48x), not a percentage change (+47.7%): the two are
easy to confuse, and `EDA.ipynb` §5.2 prints the same ratio as `+147.7%`.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (13, 4.5)
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.titleweight"] = "bold"
plt.rcParams["savefig.dpi"] = 150
plt.rcParams["savefig.bbox"] = "tight"

DATA = Path("../data/dunnhumby - Breakfast at the Frat.xlsx")
FIG = Path("../figures")
FIG.mkdir(exist_ok=True)

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
pd.set_option("display.max_columns", 50)

# palette carried over from 1_distribution_analysis.ipynb
PROMO_COL = "#e8b33a"
RED, BLUE = "#c44e52", "#4c72b0"

print(f"figures -> {FIG.resolve()}")

## 1. Value up, volume flat

Both series are indexed to their first week so they share one axis. Only `SPEND` gets a
fitted line: the `UNITS` slope is not distinguishable from zero once the 52-week cycle is
absorbed (season-adjusted p = 0.78), and drawing a trend through it would assert a
decline the data does not support. The index gap between the two lines is spend growth,
not a divergence between volume and value.


In [ ]:
tx = pd.read_excel(DATA, "dh Transaction Data", header=1)

weekly = (tx.groupby("WEEK_END_DATE")
            .agg(UNITS=("UNITS", "sum"), SPEND=("SPEND", "sum"))
            .asfreq("W-WED"))
t = np.arange(len(weekly))


def ols_slope(y, X, lag=4):
    """Slope of y on the first column of X, with Newey-West HAC(lag) errors."""
    X = np.column_stack([np.ones(len(y)), X])
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    resid = y - X @ beta
    XtX_inv = np.linalg.inv(X.T @ X)
    meat = (X * resid[:, None]).T @ (X * resid[:, None])
    for k in range(1, lag + 1):
        w = 1 - k / (lag + 1)
        G = (X[k:] * resid[k:, None]).T @ (X[:-k] * resid[:-k, None])
        meat += w * (G + G.T)
    return beta[1], np.sqrt((XtX_inv @ meat @ XtX_inv)[1, 1])


# same season-adjusted test as EDA §2: 51 week-of-year dummies absorb the cycle
D = pd.get_dummies(weekly.index.isocalendar().week, drop_first=True).to_numpy(dtype=float)

fits = {}
for col in ["UNITS", "SPEND"]:
    y = weekly[col].to_numpy(dtype=float)
    b_adj, se_adj = ols_slope(y, np.column_stack([t, D]))
    p_adj = 2 * stats.t.sf(abs(b_adj / se_adj), len(y) - 2 - D.shape[1])
    slope, _ = np.polyfit(t, y, 1)          # the headline slope, as EDA §2 prints it
    fits[col] = {"per_year": slope * 52 / y.mean(), "p": p_adj}
    print(f"{col:6s} {slope * 52 / y.mean():+.2%}/yr  season-adjusted p = {p_adj:.3f}"
          f"  -> {'trend' if p_adj < 0.05 else 'no detectable trend'}")

fig, ax = plt.subplots()
for col, colour in [("UNITS", BLUE), ("SPEND", RED)]:
    idx = weekly[col] / weekly[col].iloc[0] * 100
    ax.plot(weekly.index, idx, color=colour, lw=0.9, alpha=0.45, label=f"{col} weekly total")
    if fits[col]["p"] < 0.05:
        slope, intercept = np.polyfit(t, idx, 1)
        ax.plot(weekly.index, intercept + slope * t, color=colour, lw=2.2,
                label=f"{col} trend  {fits[col]['per_year']:+.2%}/yr  (p = {fits[col]['p']:.3f})")

ax.text(0.99, 0.06, f"UNITS: no trend detectable\n({fits['UNITS']['per_year']:+.2%}/yr, "
        f"p = {fits['UNITS']['p']:.2f})", transform=ax.transAxes, ha="right", va="bottom",
        fontsize=9, color=BLUE)
ax.axhline(100, color="gray", lw=1, ls=":")
ax.set_ylabel("index, first week = 100")
ax.set_title("Weekly units and spend: value rises, volume does not")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.10), ncols=2, frameon=False, fontsize=10)
fig.savefig(FIG / "01_units_vs_spend.png")
plt.show()


## 2. Promotion economics

The ratio the README quotes: discount given up against incremental spend, computed
exactly as [`EDA.ipynb`](EDA.ipynb) §5.2 does — a median paired SKU–store spend ratio
applied to each promo row's own non-promo baseline.

> The dataset has no cost or margin field, so this is revenue against discount. It is
> **not** a profit statement, and the chart says so on the axis.

In [ ]:
tx["ANY_PROMO"] = ((tx.DISPLAY == 1) | (tx.FEATURE == 1) | (tx.TPR_ONLY == 1)).astype(int)

pr = tx[(tx.PRICE > 0) & (tx.BASE_PRICE > 0)].copy()
pr["DISCOUNT"] = (pr.BASE_PRICE - pr.PRICE).clip(lower=0)
pr["DISC_SPEND"] = pr.DISCOUNT * pr.UNITS

cell = (pr.groupby(["UPC", "STORE_NUM", "ANY_PROMO"])
          .agg(units=("UNITS", "mean"), spend=("SPEND", "mean"), n=("UNITS", "size"))
          .unstack("ANY_PROMO"))
usable = (cell[("units", 0)] > 0) & (cell[("n", 0)] >= 4) & (cell[("n", 1)] >= 1)
unit_ratio = cell[("units", 1)][usable] / cell[("units", 0)][usable]
spend_ratio = cell[("spend", 1)][usable] / cell[("spend", 0)][usable]

baseline = cell[("spend", 0)][usable]
baseline = baseline[baseline > 0]
promo_rows = pr[pr.ANY_PROMO == 1].set_index(["UPC", "STORE_NUM"])
credit = pd.Series(promo_rows.index.map(baseline), index=promo_rows.index)
matched = credit.notna()

discount_cost = pr.loc[pr.ANY_PROMO == 1, "DISC_SPEND"].sum()
incr_spend = (credit[matched] * (spend_ratio.median() - 1)).sum()
ratio = incr_spend / discount_cost

print(f"usable cells: {usable.sum():,} of {len(usable):,}")
print(f"median paired ratio, units: {unit_ratio.median():.2f}x")
print(f"median paired ratio, spend: {spend_ratio.median():.2f}x")
print(f"discount given up:        ${discount_cost:>14,.0f}")
print(f"incremental spend:        ${incr_spend:>14,.0f}")
print(f"ratio:                              {ratio:.2f}x")

# cluster bootstrap over the same cells, as in EDA §5.2
disc_per_cell = (pr.loc[pr.ANY_PROMO == 1]
                   .groupby(["UPC", "STORE_NUM"]).DISC_SPEND.sum()
                   .reindex(baseline.index).fillna(0.0))
boot = pd.DataFrame({"baseline": baseline,
                     "n_promo": cell[("n", 1)][usable],
                     "lift": spend_ratio,
                     "discount": disc_per_cell}).dropna()
b = boot.baseline.to_numpy()
n_promo = boot.n_promo.to_numpy()
lift_v = boot.lift.to_numpy()
disc_v = boot.discount.to_numpy()

rng = np.random.default_rng(0)
draws = np.empty(2_000)
for i in range(len(draws)):
    idx = rng.integers(0, len(boot), len(boot))
    draws[i] = ((np.median(lift_v[idx]) - 1) * (b[idx] * n_promo[idx]).sum()
                / disc_v[idx].sum())
lo, hi = np.percentile(draws, [2.5, 97.5])
print(f"cluster bootstrap: 95% CI [{lo:.3f}, {hi:.3f}], "
      f"P(ratio < 1) = {np.mean(draws < 1):.3f}")

fig, ax = plt.subplots(figsize=(8, 4.8))
bars = ax.bar(["Discount given up\n(revenue forgone)", "Incremental spend\n(revenue gained)"],
              [discount_cost, incr_spend], color=[RED, PROMO_COL], width=0.55)
for b, v in zip(bars, [discount_cost, incr_spend]):
    ax.text(b.get_x() + b.get_width() / 2, v, f"${v/1e6:,.2f}M",
            ha="center", va="bottom", fontsize=12, fontweight="bold")

ax.set_ylim(0, max(discount_cost, incr_spend) * 1.12)
ax.set_yticks(np.arange(0, max(discount_cost, incr_spend) * 1.15, 1_000_000))
ax.yaxis.set_major_formatter(lambda x, _: f"${x/1e6:,.0f}M")
ax.set_title(f"Revenue return to promotion: {ratio:.2f}x  "
             f"(95% CI {lo:.2f}-{hi:.2f})")
ax.set_xlabel("no cost or margin field exists in this dataset — revenue against discount, not profit",
              fontsize=9, style="italic", color="dimgray")
fig.savefig(FIG / "02_promotion_economics.png")
plt.show()

## 3. Baskets: the receipt grows, the basket does not

Store-week aggregates, medians by ISO year. 2012 is a single week and is excluded —
see `EDA.ipynb` §2; the y-axes are zoomed so the direction is visible; the exact change
over the panel is printed above each panel.

In [ ]:
sw = (tx.groupby(["STORE_NUM", "WEEK_END_DATE"])
        .agg(spend=("SPEND", "sum"), units=("UNITS", "sum"),
             baskets=("VISITS", "sum"))
        .reset_index())
sw["per_basket"] = sw.spend / sw.baskets
sw["units_per_basket"] = sw.units / sw.baskets
sw["ISO_YEAR"] = sw.WEEK_END_DATE.dt.isocalendar().year.astype(int)

by_year = sw[sw.ISO_YEAR < 2012].groupby("ISO_YEAR")[["per_basket", "units_per_basket"]].median()
print(by_year.round(2).T)
print("change 2011 vs 2009: "
      f"spend/basket {(by_year.per_basket.iloc[-1] / by_year.per_basket.iloc[0] - 1):+.1%}, "
      f"units/basket {(by_year.units_per_basket.iloc[-1] / by_year.units_per_basket.iloc[0] - 1):+.1%}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for ax, col, colour, title in [
        (axes[0], "per_basket", RED, "Spend per basket"),
        (axes[1], "units_per_basket", BLUE, "Units per basket")]:
    change = by_year[col].iloc[-1] / by_year[col].iloc[0] - 1
    fmt = "{:,.2f}" if col == "per_basket" else "{:,.3f}"
    ax.plot(by_year.index, by_year[col], marker="o", color=colour, lw=2, ms=7)
    for x, y in by_year[col].items():
        ax.annotate(fmt.format(y), (x, y), textcoords="offset points", xytext=(0, 10),
                    ha="center", fontsize=10, fontweight="bold", color=colour)
    ax.set_xticks(by_year.index)
    ax.set_xlabel("ISO year")
    ax.set_title(f"{title} — median store-week, {change:+.1%} over the panel")
    ax.margins(y=0.20)

fig.suptitle("The receipt grows while the basket shrinks", y=1.03, fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig(FIG / "03_basket_economics.png")
plt.show()

## 4. Household acquisition

`EDA.ipynb` §5.5 recomputed. Three things are counted here: how far a promo week moves
reach (`HHS`) and depth (units per buying household), what one more buying household
costs by category after the calendar adjustment, and which promotion type actually buys
them. The cost uses the same cluster bootstrap as §5.5 — over cells, not rows.

The panel cannot follow a household, so "one more buying household" means one more
household buying that UPC in that store-week, above what the cell's own non-promo weeks
would predict.


In [ ]:
# Same pipeline as EDA §5.5: split a promo week into reach (HHS) and depth
# (units per buying household), then price the incremental households.
products = pd.read_excel(DATA, "dh Products Lookup", header=1)
hh = pr.merge(products[["UPC", "CATEGORY"]], on="UPC", how="left")
hh = hh[(hh.UNITS > 0) & (hh.HHS > 0)].copy()
hh["CELL"] = hh.STORE_NUM.astype(str) + "_" + hh.UPC.astype(str)
hh["log_HHS"] = np.log(hh.HHS)

n = hh.groupby(["CELL", "ANY_PROMO"]).size().unstack("ANY_PROMO")
paired = n[0].fillna(0).ge(8) & n[1].fillna(0).ge(8)

season = hh.groupby("WEEK_END_DATE").UNITS.sum()
season = season / season.mean()
hh["SEAS"] = hh.WEEK_END_DATE.map(season)

agg = (hh.groupby(["CELL", "ANY_PROMO"])
         .agg(hhs=("HHS", "mean"), n=("HHS", "size"), seas=("SEAS", "mean"))
         .unstack("ANY_PROMO")[paired])
disc = (hh.loc[hh.ANY_PROMO == 1].groupby("CELL").DISC_SPEND.sum()
          .reindex(agg.index).fillna(0.0))
extra = (agg[("hhs", 1)] - agg[("hhs", 0)] * agg[("seas", 1)] / agg[("seas", 0)]) * agg[("n", 1)]
cat = hh.groupby("CELL").CATEGORY.first().reindex(agg.index)


def cost_interval(incr, spend, draws=2_000, seed=0):
    """Ratio of totals, bootstrapped over cells."""
    incr, spend = np.asarray(incr), np.asarray(spend)
    rng = np.random.default_rng(seed)
    out = np.empty(draws)
    for i in range(draws):
        idx = rng.integers(0, len(incr), len(incr))
        out[i] = spend[idx].sum() / incr[idx].sum()
    return spend.sum() / incr.sum(), *np.percentile(out, [2.5, 97.5])


overall = cost_interval(extra, disc)
print(f"paired cells: {paired.sum():,} | discount ${disc.sum():,.0f}")
print(f"all cells  ${overall[0]:.2f}  [{overall[1]:.2f}, {overall[2]:.2f}]")

rows = []
for name in cat.value_counts().index:
    pick = (cat == name).to_numpy()
    point, lo, hi = cost_interval(extra.to_numpy()[pick], disc.to_numpy()[pick])
    rows.append({"category": name, "cost": point, "lo": lo, "hi": hi})
    print(f"  {name:22s} ${point:.2f}  [{lo:.2f}, {hi:.2f}]")
costs = pd.DataFrame(rows).sort_values("cost").set_index("category")


def twoway_fe(y, x, cell, week, iters=500, tol=1e-11):
    """Cell and week fixed effects by alternating projections; SE clustered on the cell."""
    y = np.asarray(y, dtype=float).copy()
    x = np.asarray(x, dtype=float).copy()
    y -= y.mean()
    x -= x.mean()
    cc, wc = pd.factorize(cell)[0], pd.factorize(week)[0]
    for _ in range(iters):
        prev_y, prev_x = y.copy(), x.copy()
        for codes in (cc, wc):
            counts = np.bincount(codes)
            y -= (np.bincount(codes, weights=y) / counts)[codes]
            x -= (np.bincount(codes, weights=x) / counts)[codes]
        if max(np.abs(y - prev_y).max(), np.abs(x - prev_x).max()) < tol:
            break
    beta = (x @ y) / (x @ x)
    resid = y - beta * x
    score = np.bincount(cc, weights=x * resid)
    n_cells = len(np.unique(cc))
    k = 1 + n_cells + len(np.unique(wc)) - 1
    dof = (n_cells / (n_cells - 1)) * ((len(y) - 1) / (len(y) - k))
    return beta, np.sqrt((score ** 2).sum() * dof) / (x @ x), len(y) - k


hh["MECH"] = "none"
hh.loc[hh.TPR_ONLY == 1, "MECH"] = "TPR only"
hh.loc[(hh.FEATURE == 1) & (hh.DISPLAY == 0), "MECH"] = "feature only"
hh.loc[(hh.DISPLAY == 1) & (hh.FEATURE == 0), "MECH"] = "display only"
hh.loc[(hh.FEATURE == 1) & (hh.DISPLAY == 1), "MECH"] = "feature+display"
hh.loc[hh.ANY_PROMO == 0, "MECH"] = "none"

mech = {}
print("reach lift by mechanism (cell + week FE):")
for name in ["TPR only", "display only", "feature only", "feature+display"]:
    d = (hh.MECH == name).to_numpy().astype(float)
    b, se, dof = twoway_fe(hh.log_HHS, d, hh.CELL, hh.WEEK_END_DATE)
    t = stats.t.ppf(0.975, dof)
    mech[name] = (np.expm1(b), np.expm1(b - t * se), np.expm1(b + t * se))
    print(f"  {name:18s} {np.expm1(b):+7.1%}  "
          f"[{np.expm1(b - t * se):+.1%}, {np.expm1(b + t * se):+.1%}]")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

ax = axes[0]
ypos = np.arange(len(costs))
ax.errorbar(costs.cost, ypos,
            xerr=[costs.cost - costs.lo, costs.hi - costs.cost],
            fmt="o", color=BLUE, ecolor=BLUE, capsize=4, ms=8, lw=1.5)
for y, (_, row) in enumerate(costs.iterrows()):
    ax.annotate(f"${row.cost:,.2f}", (row.hi, y), xytext=(8, 0),
                textcoords="offset points", va="center",
                fontsize=10, fontweight="bold", color=BLUE)
ax.set_yticks(ypos, costs.index)
ax.set_xlabel("discount per incremental buying household ($)")
ax.set_title(f"One more household costs ${overall[0]:,.2f} on average")
ax.margins(x=0.18)

ax = axes[1]
names = list(mech)
vals = np.array([mech[m][0] * 100 for m in names])
errs = np.array([[(mech[m][0] - mech[m][1]) * 100 for m in names],
                 [(mech[m][2] - mech[m][0]) * 100 for m in names]])
ax.barh(np.arange(len(names)), vals, color=PROMO_COL, edgecolor="black", lw=0.6)
ax.errorbar(vals, np.arange(len(names)), xerr=errs, fmt="none",
            ecolor="black", capsize=4, lw=1.2)
for y, v in enumerate(vals):
    ax.annotate(f"{v:+.0f}%", (v, y), xytext=(7, 0), textcoords="offset points",
                va="center", fontsize=10, fontweight="bold")
ax.set_yticks(np.arange(len(names)), names)
ax.set_xlabel("reach lift against the cell's own non-promo weeks (%), cell + week FE")
ax.set_title("Reach by promotion type")
ax.margins(x=0.16)

fig.suptitle("A promotion buys households; the price cut is the weakest way to buy them",
             fontsize=13, fontweight="bold")
fig.tight_layout()
fig.savefig(FIG / "04_household_cost.png")
plt.show()
print(f"wrote {FIG / '04_household_cost.png'}")

## 5. The calendar moves more than the trend

The same classical decomposition as [`EDA.ipynb`](EDA.ipynb) §3, drawn as a 52-week
profile so the business question is visible: how much of a normal week's volume is
calendar rather than drift? The permutation test shuffles the detrended series against
the week labels 2,000 times; the seasonal share those shuffles reach is the floor a
three-cycle average produces on its own.

![Seasonality](../figures/05_seasonality.png)

The profile is quoted as a percentage above or below the fitted trend. It is the reason a
promo test has to compare like weeks: a peak week and a trough week are not a like-for-like
baseline.

In [ ]:
# Same decomposition as EDA §3: additive on log1p, 52-week centred moving
# average, then the mean of the detrended values at each phase.
def classical_decomp(y, period=52):
    """Return (trend, seasonal, residual) for a log1p weekly series."""
    y = np.asarray(y, dtype=float)
    n = len(y)
    trend = pd.Series(y).rolling(period, center=True, min_periods=period // 2).mean()
    detrended = y - trend
    seasonal = np.array([np.nanmean(detrended[i::period]) for i in range(period)])
    seasonal -= seasonal.mean()
    seasonal = np.tile(seasonal, int(np.ceil(n / period)))[:n]
    return trend, seasonal, y - trend - seasonal


def seasonal_share(detrended, period=52):
    """Variance of the phase means — the part of the series the calendar explains."""
    phase = np.array([np.nanmean(detrended[i::period]) for i in range(period)])
    phase -= phase.mean()
    return np.var(phase)


profiles, rows = {}, []
rng = np.random.default_rng(0)          # same seed and order as EDA §3
for col in ["UNITS", "SPEND"]:
    y = np.log1p(weekly[col].to_numpy(dtype=float))
    trend_c, seasonal, resid = classical_decomp(y)
    observed = seasonal_share(y - trend_c) / np.var(y)
    null = np.array([seasonal_share(rng.permutation(y - trend_c)) / np.var(y)
                     for _ in range(2_000)])
    amp = seasonal.max() - seasonal.min()
    profiles[col] = 100 * np.expm1(seasonal[:52])
    rows.append({
        "series": col,
        "seasonal %": 100 * np.var(seasonal) / np.var(y),
        "trend %": 100 * np.nanvar(trend_c) / np.var(y),
        "peak-to-trough %": 100 * np.expm1(amp),
        "shuffle p": np.mean(null >= observed),
        "shuffles >= observed": int((null >= observed).sum()),
    })

seasonality = pd.DataFrame(rows).set_index("series")
print(seasonality.drop(columns="shuffle p").round(2).to_string())
print("\npermutation: 2,000 shuffles per series; p is the share that match or beat "
      "the observed seasonal share")
for col, row in seasonality.iterrows():
    print(f"  {col:6s} shuffle p {row['shuffle p']:.4f} "
          f"({int(row['shuffles >= observed']):,} of 2,000)")

fig, ax = plt.subplots(figsize=(13, 4.4))
weeks = np.arange(1, 53)
for col, colour in [("UNITS", BLUE), ("SPEND", RED)]:
    ax.plot(weeks, profiles[col], color=colour, lw=2, label=col)
    peak, trough = int(np.argmax(profiles[col])), int(np.argmin(profiles[col]))
    ax.scatter([peak + 1, trough + 1], [profiles[col][peak], profiles[col][trough]],
               color=colour, s=45, zorder=4)
    ax.annotate(f"peak +{profiles[col][peak]:.0f}%", (peak + 1, profiles[col][peak]),
                xytext=(8, 7), textcoords="offset points", fontsize=9,
                fontweight="bold", color=colour)
    ax.annotate(f"trough {profiles[col][trough]:.0f}%", (trough + 1, profiles[col][trough]),
                xytext=(8, -13), textcoords="offset points", fontsize=9,
                fontweight="bold", color=colour)

ax.axhline(0, color="gray", lw=1, ls=":")
ax.set_xticks([1, 13, 26, 39, 52], ["Jan", "Apr", "Jul", "Oct", "Dec"])
ax.set_xlim(1, 52)
ax.set_ylabel("% vs fitted trend")
ax.set_xlabel("week of year")
ax.set_title("Seasonality dominates the trend: the calendar is the bigger lever")
ax.legend(loc="upper left", frameon=False, ncols=2, fontsize=10)
fig.savefig(FIG / "05_seasonality.png")
plt.show()
print(f"wrote {FIG / '05_seasonality.png'}")

## Regenerating

*Run All* rewrites the five PNGs in `../figures/`. The notebook is committed without
outputs; the figures are committed as files, because GitHub does not render notebook
outputs inside a README.

If a number printed here disagrees with the section of `EDA.ipynb` named in the table
at the top, the figure is out of date — fix whichever side is wrong, do not edit the
caption to match.